# Addressing Reviewer Comments 1

This notebook contains the code to replicate select results from the [paper on arxiv](https://arxiv.org/abs/2504.06195).

# old code

## imports & load data

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cross_decomposition import PLSRegression

from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA

from tqdm import tqdm
import scipy.stats as stats

from whecho.whecho import whecho_simple
import pickle
import os


### load matrices

In [2]:
# TiTE
all_data = {}
all_data["Ti_TE"] = pd.read_excel("../p04_TE_TM_1D_compare/Ti_TE_matrix.xlsx", header=None).transpose()
all_data["Ti_TE"].columns = [f"feature_{i+1}" for i in range(all_data["Ti_TE"].shape[1])]
all_data["Ti_TE"]["target"] = list(range(1, all_data["Ti_TE"].shape[0] + 1))

In [3]:
all_data["Ti_TE"].shape

(101, 10001)

In [4]:
for dat in ["Si_TE", "Si_TM", "Ti_TM"]:
    all_data[dat] = pd.read_excel(f"../p06_multvar_SI_fitting/{dat}_matrix.xlsx", header=None).transpose()
    all_data[dat].columns = [f"feature_{i+1}" for i in range(all_data[dat].shape[1])]
    all_data[dat]["target"] = list(range(1, all_data[dat].shape[0] + 1))

### build the 5-fold cross validation groups and construct PCs

In [5]:
# split the data into 5-folds cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=19890417)
#kf = KFold(n_splits=5, shuffle=True, random_state=42)
PC_folds = {}
folds = {} # this will not use PC to reduce the data

for dat in all_data.keys():
    PC_folds[dat] = []
    folds[dat] = []
    full_data = all_data[dat].copy()
    for train_index, test_index in kf.split(all_data[dat]):
        # because this doesn't check for out of range values we need to make sure that we move any 0 or 100 index values from test to train
        # if 0 in test_index:
        #     test_index = np.delete(test_index, np.where(test_index == 0)[0][0])
        #     train_index = np.append(train_index, 0)
        # if 100 in test_index:
        #     test_index = np.delete(test_index, np.where(test_index == 100)[0][0])
        #     train_index = np.append(train_index, 100)
        # train_index.sort()
        # print(f"train: {train_index}, test: {test_index}")
        train_data = full_data.iloc[train_index]
        test_data = full_data.iloc[test_index]
        # first put the raw data into the folds
        folds[dat].append((train_data.drop(columns=["target"]), train_data["target"].values, test_data.drop(columns=["target"]), test_data["target"].values))
        # now we also want to dimensionally reduce the data into 80 PCs following the training data
        pca = PCA(n_components=80, svd_solver='full', random_state=19890417)
        # first standardize the data according to the training data
        scaler = StandardScaler()
        scaler.fit(train_data.drop(columns=["target"]))
        train_data_scaled = scaler.transform(train_data.drop(columns=["target"]))
        test_data_scaled = scaler.transform(test_data.drop(columns=["target"]))
        pca.fit(train_data_scaled)
        train_data_pca = pca.transform(train_data_scaled)
        test_data_pca = pca.transform(test_data_scaled)
        PC_folds[dat].append((train_data_pca, train_data["target"].values, test_data_pca, test_data["target"].values))

## Identify best single variate performer for each dataset

In [6]:
# first check if the results_dfs.pickle file exists and if so load it
if os.path.exists("results_dfs.pickle"):
    with open("results_dfs.pickle", "rb") as f:
        results_dfs = pickle.load(f)
        print("Loaded existing results_dfs from pickle file.")
else:
    results_dfs = {}
    for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
        result_list = []
        fold_i = 0
        for train_data_pca, train_target, test_data_pca, test_target in folds[dat]:
            for feat_i in tqdm(range(train_data_pca.shape[1])):
                X_train = train_data_pca[f"feature_{feat_i+1}"].to_numpy()
                y_train = train_target
                X_test = test_data_pca[f"feature_{feat_i+1}"].to_numpy()
                y_test = test_target
                # Fit a linear regression model
                model = LinearRegression()
                model.fit(X_train.reshape(-1, 1), y_train)
                # Make predictions
                y_pred = model.predict(X_test.reshape(-1, 1))
                # Calculate MSE, R2, RMSE, MAE
                mse = mean_squared_error(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)
                rmse = np.sqrt(mse)
                mae = np.mean(np.abs(y_test - y_pred))
                # now calculate the same metrics on the training data
                y_train_pred = model.predict(X_train.reshape(-1, 1))
                mse_train = mean_squared_error(y_train, y_train_pred)
                r2_train = r2_score(y_train, y_train_pred)
                rmse_train = np.sqrt(mse_train)
                mae_train = np.mean(np.abs(y_train - y_train_pred))
                # Append results to the list
                result_list.append({
                    "feature": f"feature_{feat_i+1}",
                    "mse": mse,
                    "r2": r2,
                    "rmse": rmse,
                    "mae": mae,
                    "fold": fold_i,
                    "mse.train": mse_train,
                    "r2.train": r2_train,
                    "rmse.train": rmse_train,
                    "mae.train": mae_train,
                })
            fold_i += 1
        results_dfs[dat] = pd.DataFrame(result_list)
    # save results_dfs to a pickle file
    with open("results_dfs.pickle", "wb+") as f:
        pickle.dump(results_dfs, f)

Loaded existing results_dfs from pickle file.


In [7]:
# for each dataset we want to print the mean and std of the metrics for the best feature
# also save them to a lookup table
best_results = {}
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    print(f"Dataset: {dat}")
    # first group by feature to find the feature with lowest mean mse
    mean_results = results_dfs[dat].groupby("feature").agg({
        "mse": ["mean","std"],
        "r2": ["mean","std"],
        "rmse": ["mean","std"],
        "mae": ["mean","std"],
        "mse.train": ["mean","std"],
        "r2.train": ["mean","std"],
        "rmse.train": ["mean","std"],
        "mae.train": ["mean","std"],
    })
    mean_results.columns = ["_".join(col).strip() for col in mean_results.columns.values]
    mean_results = mean_results.reset_index()
    # now we want to find the feature with the lowest mean mse for training
    best_feature_idx = mean_results["mse.train_mean"].idxmin()
    best_feature = mean_results.iloc[best_feature_idx]
    print(f"Best feature: {best_feature['feature']} (train MSE: {best_feature['mse.train_mean']:.4f} std ({best_feature['mse.train_std']:.4f}))")
    print(f"Mean MSE: {best_feature['mse_mean']:.4f} std ({best_feature['mse_std']:.4f})")
    print(f"Mean R2: {best_feature['r2_mean']:.4f} std ({best_feature['r2_std']:.4f})")
    print(f"Mean RMSE: {best_feature['rmse_mean']:.4f} std ({best_feature['rmse_std']:.4f})")
    print(f"Mean MAE: {best_feature['mae_mean']:.4f} std ({best_feature['mae_std']:.4f})")
    print()
    # save the best feature to the lookup table
    best_results[dat] = {
        "feature": best_feature["feature"],
        "mse_mean": best_feature["mse_mean"],
        "mse_std": best_feature["mse_std"],
        "r2_mean": best_feature["r2_mean"],
        "r2_std": best_feature["r2_std"],
        "rmse_mean": best_feature["rmse_mean"],
        "rmse_std": best_feature["rmse_std"],
        "mae_mean": best_feature["mae_mean"],
        "mae_std": best_feature["mae_std"],
        "mse.train_mean": best_feature["mse.train_mean"],
        "mse.train_std": best_feature["mse.train_std"], 
        "r2.train_mean": best_feature["r2.train_mean"],
        "r2.train_std": best_feature["r2.train_std"],
        "rmse.train_mean": best_feature["rmse.train_mean"],
        "rmse.train_std": best_feature["rmse.train_std"],
        "mae.train_mean": best_feature["mae.train_mean"],
        "mae.train_std": best_feature["mae.train_std"],
    }

Dataset: Ti_TM
Best feature: feature_5094 (train MSE: 0.4017 std (0.0247))
Mean MSE: 0.4462 std (0.1050)
Mean R2: 0.9994 std (0.0002)
Mean RMSE: 0.6636 std (0.0855)
Mean MAE: 0.5234 std (0.0653)

Dataset: Ti_TE
Best feature: feature_4956 (train MSE: 0.0628 std (0.0038))
Mean MSE: 0.0655 std (0.0150)
Mean R2: 0.9999 std (0.0000)
Mean RMSE: 0.2547 std (0.0284)
Mean MAE: 0.2151 std (0.0306)

Dataset: Si_TE
Best feature: feature_390 (train MSE: 0.2336 std (0.0204))
Mean MSE: 0.2416 std (0.0866)
Mean R2: 0.9997 std (0.0001)
Mean RMSE: 0.4853 std (0.0873)
Mean MAE: 0.4001 std (0.0558)

Dataset: Si_TM
Best feature: feature_1206 (train MSE: 0.1355 std (0.0089))
Mean MSE: 0.1417 std (0.0378)
Mean R2: 0.9998 std (0.0001)
Mean RMSE: 0.3736 std (0.0509)
Mean MAE: 0.3097 std (0.0525)



## Now Evaluate the performance when the entire peak is used (Figure 7)

In [8]:
results_entire_wave_dfs = {}
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        # now we want to use the entire wave data
        # Fit a linear regression model
        model = LinearRegression()
        model.fit(train_data_pca, train_target)
        # Make predictions
        y_pred = model.predict(test_data_pca)
        # Calculate MSE, R2, RMSE, MAE
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        # Append results to the list
        result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_entire_wave_dfs[dat] = pd.DataFrame(result_list)

# for each dataset we want to print the mean and std of the metrics for the best feature
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    print(f"Dataset: {dat}")
    # first group by feature to find the feature with lowest mean mse
    mean_results = results_entire_wave_dfs[dat].agg({
        "mse": ["mean","std"],
        "r2": ["mean","std"],
        "rmse": ["mean","std"],
        "mae": ["mean","std"],
    })
    # mean_results.columns = ["_".join(col).strip() for col in mean_results.columns.values]
    
    # just print the mean and std
    # calculate the fold improvement for mse compared to the best feature
    fold_improvement = (mean_results["mse"].values[0] / best_results[dat]["mse_mean"])**-1
    print(f"Mean MSE: {mean_results['mse'].values[0]:.4e} std ({mean_results['mse'].values[1]:.4e}). {fold_improvement:.2f}x fold improvement")
    fold_improvement = (mean_results["r2"].values[0] / best_results[dat]["r2_mean"])**-1
    print(f"Mean R2: {mean_results['r2'].values[0]:.4f} std ({mean_results['r2'].values[1]:.4f}). {fold_improvement:.2f}x fold improvement")
    fold_improvement = (mean_results["rmse"].values[0] / best_results[dat]["rmse_mean"])**-1
    print(f"Mean RMSE: {mean_results['rmse'].values[0]:.4f} std ({mean_results['rmse'].values[1]:.4f}). {fold_improvement:.2f}x fold improvement")
    fold_improvement = (mean_results["mae"].values[0] / best_results[dat]["mae_mean"])**-1
    print(f"Mean MAE: {mean_results['mae'].values[0]:.4f} std ({mean_results['mae'].values[1]:.4f}). {fold_improvement:.2f}x fold improvement")
    print()

Dataset: Ti_TM
Mean MSE: 6.6233e-05 std (2.1992e-05). 6736.42x fold improvement
Mean R2: 1.0000 std (0.0000). 1.00x fold improvement
Mean RMSE: 0.0080 std (0.0014). 82.52x fold improvement
Mean MAE: 0.0066 std (0.0010). 79.74x fold improvement

Dataset: Ti_TE
Mean MSE: 1.0488e-04 std (6.4012e-05). 624.73x fold improvement
Mean R2: 1.0000 std (0.0000). 1.00x fold improvement
Mean RMSE: 0.0099 std (0.0031). 25.82x fold improvement
Mean MAE: 0.0072 std (0.0016). 29.84x fold improvement

Dataset: Si_TE
Mean MSE: 2.4732e-02 std (9.9685e-03). 9.77x fold improvement
Mean R2: 1.0000 std (0.0000). 1.00x fold improvement
Mean RMSE: 0.1547 std (0.0314). 3.14x fold improvement
Mean MAE: 0.1257 std (0.0288). 3.18x fold improvement

Dataset: Si_TM
Mean MSE: 1.0345e-01 std (2.1553e-01). 1.37x fold improvement
Mean R2: 0.9999 std (0.0003). 1.00x fold improvement
Mean RMSE: 0.2035 std (0.2785). 1.84x fold improvement
Mean MAE: 0.0871 std (0.0670). 3.56x fold improvement



## check the sv Si when using the peak shifts 1D datasets (Table 1)

In [9]:
Si_TE_1D = pd.read_excel("../p07_1D_SI_fitting/Si_TE_1Dfitting_4peaks.xlsx", header=None)
Si_TE_1D.columns = ["target","feature_1", "feature_2", "feature_3", "feature_4"]

Si_TM_1D = pd.read_excel("../p07_1D_SI_fitting/Si_TM_1Dfitting_4peaks.xlsx", header=None)
Si_TM_1D.columns = ["target","feature_1", "feature_2", "feature_3", "feature_4"]

Si_TE_1D_results = []
Si_TM_1D_results = []
# perform 5-fold cross validation on the 1D data
fold_i = 0
for train_index, test_index in kf.split(Si_TE_1D):
    for df, result in zip([Si_TE_1D, Si_TM_1D],[Si_TE_1D_results, Si_TM_1D_results]):
        train_data = df.iloc[train_index]
        test_data = df.iloc[test_index]
        for feat_name in [f"feature_{i+1}" for i in range(4)]:
            X_train = train_data[feat_name].to_numpy()
            y_train = train_data["target"].values
            X_test = test_data[feat_name].to_numpy()
            y_test = test_data["target"].values
            # Fit a linear regression model
            model = LinearRegression()
            model.fit(X_train.reshape(-1, 1), y_train)
            # Make predictions
            y_pred = model.predict(X_test.reshape(-1, 1))
            # Calculate MSE, R2, RMSE, MAE
            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = np.mean(np.abs(y_test - y_pred))
            # now calculate the same metrics on the training data
            y_train_pred = model.predict(X_train.reshape(-1, 1))
            mse_train = mean_squared_error(y_train, y_train_pred)
            r2_train = r2_score(y_train, y_train_pred)
            rmse_train = np.sqrt(mse_train)
            mae_train = np.mean(np.abs(y_train - y_train_pred))
            # Append results to the list
            result.append({
                "feature": feat_name,
                "mse": mse,
                "r2": r2,
                "rmse": rmse,
                "mae": mae,
                "fold": fold_i,
                "mse.train": mse_train,
                "r2.train": r2_train,
                "rmse.train": rmse_train,
                "mae.train": mae_train,
            })
        # now we also want to try using all 4 features
        X_train = train_data.drop(columns=["target"]).to_numpy()
        y_train = train_data["target"].values
        X_test = test_data.drop(columns=["target"]).to_numpy()
        y_test = test_data["target"].values
        # Fit a linear regression model
        model = LinearRegression()
        model.fit(X_train, y_train)
        # Make predictions
        y_pred = model.predict(X_test)
        # Calculate MSE, R2, RMSE, MAE
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
        # now calculate the same metrics on the training data
        y_train_pred = model.predict(X_train)
        mse_train = mean_squared_error(y_train, y_train_pred)
        r2_train = r2_score(y_train, y_train_pred)
        rmse_train = np.sqrt(mse_train)
        mae_train = np.mean(np.abs(y_train - y_train_pred))
        # Append results to the list
        result.append({
            "feature": "all_features",
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
            "mse.train": mse_train,
            "r2.train": r2_train,
            "rmse.train": rmse_train,
            "mae.train": mae_train,
        })
    fold_i += 1
Si_TE_1D_result_df = pd.DataFrame(Si_TE_1D_results)
Si_TM_1D_result_df = pd.DataFrame(Si_TM_1D_results)
    

In [10]:
# recall the best result using the entire wave data
# Dataset: Si_TE
# Mean MSE: 0.02473 std (9.9685e-03). ~22.7x fold improvement
# Mean R2: 1.0000 std (0.0000). 1.00x fold improvement
# Mean RMSE: 0.1547 std (0.0314). 3.14x fold improvement
# Mean MAE: 0.1257 std (0.0288). 3.18x fold improvement

Si_TE_1D_result_df.groupby("feature").agg({
    "mse": ["mean","std"],
    "r2": ["mean","std"],
    "rmse": ["mean","std"],
    "mae": ["mean","std"],
    "mse.train": ["mean","std"],
    "r2.train": ["mean","std"],
    "rmse.train": ["mean","std"],
    "mae.train": ["mean","std"],
})

mse                  r2                rmse            \
                   mean       std      mean       std      mean       std   
feature                                                                     
all_features   0.135694  0.026329  0.999824  0.000021  0.366952  0.036061   
feature_1     10.035007  1.843891  0.986622  0.003594  3.157665  0.283188   
feature_2      1.822854  0.590767  0.997550  0.000949  1.336948  0.210429   
feature_3      2.907157  0.638258  0.996132  0.001105  1.697132  0.183369   
feature_4      0.562897  0.073247  0.999250  0.000174  0.749018  0.048342   

                   mae           mse.train            r2.train            \
                  mean       std      mean       std      mean       std   
feature                                                                    
all_features  0.296414  0.037559  0.121764  0.006810  0.999856  0.000008   
feature_1     2.729086  0.278675  8.801394  0.272485  0.989567  0.000602   
feature_2     1.125622  0.127932  1.554510  0.100828  0.998158  0.000142   
feature_3     1.436463  0.196374  2.599922  0.108381  0.996920  0.000161   
feature_4     0.604117  0.032712  0.512121  0.016505  0.999392  0.000044   

             rmse.train           mae.train            
                   mean       std      mean       std  
feature                                                
all_features   0.348838  0.009749  0.280877  0.010746  
feature_1      2.966428  0.046052  2.568578  0.061155  
feature_2      1.246267  0.040747  1.057536  0.034875  
feature_3      1.612147  0.033630  1.369071  0.052450  
feature_4      0.715551  0.011602  0.582389  0.008291

In [11]:
# Recall the best result using the entire wave data
# Dataset: Si_TM
# Mean MSE: 0.1034 std (2.1553e-01). ~5x fold worsensing
# Mean R2: 0.9999 std (0.0003). ~20% worse
# Mean RMSE: 0.2035 std (0.2785). ~20% worse
# Mean MAE: 0.0871 std (0.0670). ~2x fold improvement ?

Si_TM_1D_result_df.groupby("feature").agg({
    "mse": ["mean","std"],
    "r2": ["mean","std"],
    "rmse": ["mean","std"],
    "mae": ["mean","std"],
    "mse.train": ["mean","std"],
    "r2.train": ["mean","std"],
    "rmse.train": ["mean","std"],
    "mae.train": ["mean","std"],
})

mse                  r2                rmse            \
                  mean       std      mean       std      mean       std   
feature                                                                    
all_features  0.004556  0.001969  0.999994  0.000003  0.065996  0.015823   
feature_1     0.791981  0.165257  0.998944  0.000304  0.886253  0.090394   
feature_2     5.897534  1.077055  0.992177  0.001999  2.421099  0.211578   
feature_3     0.755887  0.109434  0.998999  0.000221  0.867713  0.060837   
feature_4     0.029436  0.006827  0.999961  0.000012  0.170650  0.019834   

                   mae           mse.train            r2.train                \
                  mean       std      mean       std      mean           std   
feature                                                                        
all_features  0.053686  0.012416  0.004328  0.000483  0.999995  7.940732e-07   
feature_1     0.754197  0.086529  0.692697  0.027480  0.999179  5.231713e-05   
feature_2     2.103621  0.170806  5.226674  0.179268  0.993809  2.495997e-04   
feature_3     0.736634  0.057786  0.668325  0.015962  0.999208  2.993195e-05   
feature_4     0.137132  0.014416  0.027322  0.001810  0.999968  3.098957e-06   

             rmse.train           mae.train            
                   mean       std      mean       std  
feature                                                
all_features   0.065707  0.003633  0.051843  0.003193  
feature_1      0.832152  0.016579  0.710209  0.019945  
feature_2      2.285918  0.039535  2.005111  0.052579  
feature_3      0.817464  0.009800  0.696552  0.014512  
feature_4      0.165218  0.005523  0.132627  0.004225

## save the actual and fitted values for the best ML model for each dataset (Figure 8)

In [12]:
pred_fitted_results_entire_wave_dfs = {}
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    result_df = None
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        # now we want to use the entire wave data
        # Fit a linear regression model
        model = LinearRegression()
        model.fit(train_data_pca, train_target)
        # Make predictions
        y_pred = model.predict(test_data_pca)
        
        # now I want to create a dataframe with the true, predicted, and train/test labels
        pred_fitted_test_df = pd.DataFrame({
            "true": test_target,
            f"predicted_fold{fold_i}": y_pred,
            f"train/test_fold{fold_i}": ["test"] * len(test_target),
        })
        # now I want to create a dataframe with the true, predicted, and train/test labels for the training data
        pred_fitted_train_df = pd.DataFrame({
            "true": train_target,
            f"predicted_fold{fold_i}": model.predict(train_data_pca),
            f"train/test_fold{fold_i}": ["train"] * len(train_target),
        })
        # now I want to concatenate the two dataframes
        pred_fitted_df = pd.concat([pred_fitted_train_df, pred_fitted_test_df])
        fold_i += 1
        if result_df is None:
            result_df = pred_fitted_df
        else:
            result_df = result_df.merge(pred_fitted_df, on="true", how="outer")
    pred_fitted_results_entire_wave_dfs[dat] = result_df

In [13]:
# save the pred_fitted_results_entire_wave_dfs to a xlsx file one to a sheet
with pd.ExcelWriter("pred_fitted_results_entire_wave_dfs.xlsx") as writer:
    for dat in pred_fitted_results_entire_wave_dfs.keys():
        pred_fitted_results_entire_wave_dfs[dat].to_excel(writer, sheet_name=dat, index=False)

In [14]:
whecho_simple("starting section about the reviewer comments")

# Reviewer #1 Comment about Cross-Validation

In [15]:
# commenting to speed up nb (but all this cell does is produce the supplemental table 1 comparing the train-test splitting)
# suppl_table1_sheets = {}
# for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
#     sheet_columns = []
#     level1 = [f"Fold {i}" for i in range(1, 6) for _ in range(3)]
#     level2 = ["training mean (std)", "testing mean (std)", "Pvalue"]*5
#     columns = pd.MultiIndex.from_arrays([level1, level2], names=['fold', 'stat'])
#     indices = ["target"] + [f"feature {i+1}" for i in range(1, 10001)]
#     for train_data, train_target, test_data, test_target in folds[dat]:
#         # Compute mean and std for each feature in train and test
#         train_means = train_data.mean(axis=0)
#         train_stds = train_data.std(axis=0)
#         test_means = test_data.mean(axis=0)
#         test_stds = test_data.std(axis=0)
#         # t-test for each feature
#         ttest_pvals = []
#         for col in train_data.columns:
#             t_stat, p_val = stats.ttest_ind(train_data[col], test_data[col], equal_var=False)
#             ttest_pvals.append(p_val)
#         col1 = ["{:.2e} ({:.2e})".format(m,s) for m,s in zip(train_means.values, train_stds.values)]
#         col2 = ["{:.2e} ({:.2e})".format(m,s) for m,s in zip(test_means.values, test_stds.values)]
#         col3 = ["{:.4f}".format(p) if p > 1e-4 else "<0.0001" for p in ttest_pvals]
#         # add the mean and variance and p-value of the target value to the front of the column lists
#         col1.insert(0, "{:.2e} ({:.2e})".format(train_target.mean(), train_target.std()))
#         col2.insert(0, "{:.2e} ({:.2e})".format(test_target.mean(), test_target.std()))
#         target_pval = stats.ttest_ind(train_target, test_target, equal_var=False)[1]
#         col3.insert(0, "<0.0001" if target_pval < 1e-4 else "{:.4f}".format(target_pval))
#         sheet_columns.append(col1)
#         sheet_columns.append(col2)
#         sheet_columns.append(col3)
#     suppl_table1_sheets[dat] = pd.DataFrame(np.array(sheet_columns).T, columns=columns, index=indices)

In [17]:
# # Write with MultiIndex headers preserved using openpyxl
# df = suppl_table1_sheets[dat]

# with pd.ExcelWriter(f"suppl_table1.xlsx", engine='openpyxl') as writer:
#     for dat in suppl_table1_sheets.keys():
#         # Write the DataFrame to the specified sheet
#         suppl_table1_sheets[dat].to_excel(writer, sheet_name=f"{dat} 5-fold stats", index=True)

# Reviewer #1 Comment about best single predictor

In [18]:
with pd.ExcelWriter(f"suppl_table2.xlsx", engine='openpyxl') as writer:
    for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
        print(f"Dataset: {dat}")
        # first group by feature to find the feature with lowest mean mse
        mean_results = results_dfs[dat].groupby("feature").agg({
            "mse": ["mean","std"],
            "r2": ["mean","std"],
            "rmse": ["mean","std"],
            "mae": ["mean","std"],
            "mse.train": ["mean","std"],
            "r2.train": ["mean","std"],
            "rmse.train": ["mean","std"],
            "mae.train": ["mean","std"],
        })
        mean_results.sort_values(by=("mse","mean")).to_excel(writer, sheet_name=f"{dat} single predictor results", index=True)

Dataset: Ti_TM
Dataset: Ti_TE
Dataset: Si_TE
Dataset: Si_TM


# Reviewer #2 Comment on number of principal components

refer to the notebook in: `code\p06_multvar_SI_fitting\single_pred_vs_mult_PC.ipynb`

In [19]:
# in powershell I want to copy the file from code/p06_multvar_SI_fitting/all_PCs_mean_results.xlsx to code/p10_consolidating_reviewer_feedback/suppl_table3.xlsx
! powershell Copy-Item -Path "../p06_multvar_SI_fitting/all_PCs_mean_results.xlsx" -Destination "suppl_table3.xlsx"

# Reviewer #1 & #2 Comment about non-linear modeling

This will be difficult to respond to. Reviewer 2 highlights the following models

In [20]:
# SVR with RBF kernel
results_svr_dfs = {}
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        # No scaling needed for PCA data - PCs are already standardized
        
        # Fit SVR model with RBF kernel
        model = SVR(kernel='rbf', C=1.0, gamma='scale')
        model.fit(train_data_pca, train_target)
        # Make predictions
        y_pred = model.predict(test_data_pca)
        # Calculate metrics
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        # Append results
        result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_svr_dfs[dat] = pd.DataFrame(result_list)

In [21]:
results_svr_dfs["Ti_TM"]

,mse,r2,rmse,mae,fold
0,245.085322,0.674030,15.655201,10.692680,0
1,242.413224,0.763716,15.569625,11.554959,1
2,168.415246,0.774668,12.977490,7.741769,2
3,116.399698,0.815047,10.788869,5.604096,3
4,385.938850,0.466513,19.645326,13.190922,4


In [22]:
results_entire_wave_dfs["Ti_TM"]

,mse,r2,rmse,mae,fold
0,0.000076,1.0,0.008709,0.007681,0
1,0.000042,1.0,0.006449,0.005393,1
2,0.000082,1.0,0.009061,0.007027,2
3,0.000088,1.0,0.009380,0.007056,3
4,0.000044,1.0,0.006607,0.005666,4


In [23]:
# Neural Network (MLP Regressor)
results_nn_dfs = {}
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        # No scaling needed for PCA data
        
        # Fit Neural Network model
        model = MLPRegressor(hidden_layer_sizes=(50,25), max_iter=1000, random_state=19890417)
        model.fit(train_data_pca, train_target)
        # Make predictions
        y_pred = model.predict(test_data_pca)
        # Calculate metrics
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        # Append results
        result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_nn_dfs[dat] = pd.DataFrame(result_list)

In [24]:
results_nn_dfs["Ti_TM"]

,mse,r2,rmse,mae,fold
0,0.541053,0.999280,0.735563,0.583354,0
1,0.303976,0.999704,0.551341,0.443369,1
2,1.136061,0.998480,1.065862,0.868900,2
3,0.720975,0.998854,0.849102,0.687366,3
4,1.305240,0.998196,1.142471,0.833141,4


In [25]:
# XGBoost
results_xgb_dfs = {}
for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        # Fit XGBoost model
        model = XGBRegressor(n_estimators=100, random_state=19890417, verbosity=0)
        model.fit(train_data_pca, train_target)
        # Make predictions
        y_pred = model.predict(test_data_pca)
        # Calculate metrics
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        # Append results
        result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_xgb_dfs[dat] = pd.DataFrame(result_list)

In [26]:
results_xgb_dfs["Ti_TM"]

,mse,r2,rmse,mae,fold
0,2.536346,0.996627,1.592591,1.414469,0
1,1.634298,0.998407,1.278397,1.113264,1
2,3.402285,0.995448,1.844528,1.510601,2
3,3.116517,0.995048,1.765366,1.494625,3
4,4.639960,0.993586,2.154057,1.735904,4


Try to use optuna to determine the best hyper-parameters for each model:

In [27]:
max_trials = 100

# Optuna optimization for hyperparameters
optuna.logging.set_verbosity(optuna.logging.INFO)  # Changed from WARNING to INFO

def create_svr_objective(dataset_name, cv_folds):
    def objective(trial):
        C = trial.suggest_float('C', 0.01, 100.0, log=True)
        gamma = trial.suggest_float('gamma', 1e-6, 1.0, log=True)
        epsilon = trial.suggest_float('epsilon', 0.001, 1.0, log=True)
        
        mse_scores = []
        for train_data_pca, train_target, test_data_pca, test_target in cv_folds:
            model = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon)
            model.fit(train_data_pca, train_target)
            y_pred = model.predict(test_data_pca)
            mse = mean_squared_error(test_target, y_pred)
            mse_scores.append(mse)
        
        return np.mean(mse_scores)
    return objective

def create_nn_objective(dataset_name, cv_folds):
    def objective(trial):
        # Architecture
        n_layers = trial.suggest_int('n_layers', 1, 3)
        hidden_sizes = []
        for i in range(n_layers):
            size = trial.suggest_int(f'hidden_size_{i}', 10, 200)
            hidden_sizes.append(size)
        hidden_layer_sizes = tuple(hidden_sizes)
        
        # Other hyperparameters
        alpha = trial.suggest_float('alpha', 1e-6, 1e-2, log=True)
        learning_rate_init = trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True)
        
        mse_scores = []
        for train_data_pca, train_target, test_data_pca, test_target in cv_folds:
            model = MLPRegressor(
                hidden_layer_sizes=hidden_layer_sizes,
                alpha=alpha,
                learning_rate_init=learning_rate_init,
                max_iter=1000,
                random_state=19890417
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model.fit(train_data_pca, train_target)
            y_pred = model.predict(test_data_pca)
            mse = mean_squared_error(test_target, y_pred)
            mse_scores.append(mse)
        
        return np.mean(mse_scores)
    return objective

def create_xgb_objective(dataset_name, cv_folds):
    def objective(trial):
        n_estimators = trial.suggest_int('n_estimators', 50, 500)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
        subsample = trial.suggest_float('subsample', 0.6, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
        reg_alpha = trial.suggest_float('reg_alpha', 0.0, 1.0)
        reg_lambda = trial.suggest_float('reg_lambda', 0.0, 1.0)
        
        mse_scores = []
        for train_data_pca, train_target, test_data_pca, test_target in cv_folds:
            model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                random_state=19890417,
                verbosity=0
            )
            model.fit(train_data_pca, train_target)
            y_pred = model.predict(test_data_pca)
            mse = mean_squared_error(test_target, y_pred)
            mse_scores.append(mse)
        
        return np.mean(mse_scores)
    return objective

# Run optimization for each model and dataset
print("Starting hyperparameter optimization...")
best_params = {}

for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    print(f"\nOptimizing for dataset: {dat}")
    best_params[dat] = {}
    
    # SVR optimization
    print("  Optimizing SVR...")
    study_svr = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=19890417))
    study_svr.optimize(create_svr_objective(dat, PC_folds[dat]), n_trials=max_trials, show_progress_bar=True)
    best_params[dat]['svr'] = study_svr.best_params
    print(f"    Best SVR MSE: {study_svr.best_value:.6f}")
    
    # Neural Network optimization
    print("  Optimizing Neural Network...")
    study_nn = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=19890417))
    study_nn.optimize(create_nn_objective(dat, PC_folds[dat]), n_trials=max_trials, show_progress_bar=True)
    best_params[dat]['nn'] = study_nn.best_params
    print(f"    Best NN MSE: {study_nn.best_value:.6f}")
    
    # XGBoost optimization
    print("  Optimizing XGBoost...")
    study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=19890417))
    study_xgb.optimize(create_xgb_objective(dat, PC_folds[dat]), n_trials=max_trials, show_progress_bar=True)
    best_params[dat]['xgb'] = study_xgb.best_params
    print(f"    Best XGBoost MSE: {study_xgb.best_value:.6f}")

# Train final models with optimized parameters
results_svr_optimized_dfs = {}
results_nn_optimized_dfs = {}
results_xgb_optimized_dfs = {}

print("\nTraining final models with optimized parameters...")

for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    print(f"\nTraining optimized models for {dat}...")
    
    # SVR with optimized parameters
    svr_result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        params = best_params[dat]['svr']
        model = SVR(kernel='rbf', **params)
        model.fit(train_data_pca, train_target)
        y_pred = model.predict(test_data_pca)
        
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        
        svr_result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_svr_optimized_dfs[dat] = pd.DataFrame(svr_result_list)
    
    # Neural Network with optimized parameters
    nn_result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        params = best_params[dat]['nn'].copy()
        # Reconstruct hidden_layer_sizes tuple
        n_layers = len([k for k in params.keys() if k.startswith('hidden_size_')])
        hidden_sizes = []
        for i in range(n_layers):
            hidden_sizes.append(params.pop(f'hidden_size_{i}'))
        params['hidden_layer_sizes'] = tuple(hidden_sizes)
        params.pop('n_layers', None)
        
        model = MLPRegressor(max_iter=1000, random_state=19890417, **params)
        model.fit(train_data_pca, train_target)
        y_pred = model.predict(test_data_pca)
        
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        
        nn_result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_nn_optimized_dfs[dat] = pd.DataFrame(nn_result_list)
    
    # XGBoost with optimized parameters
    xgb_result_list = []
    fold_i = 0
    for train_data_pca, train_target, test_data_pca, test_target in PC_folds[dat]:
        params = best_params[dat]['xgb']
        model = XGBRegressor(random_state=19890417, verbosity=0, **params)
        model.fit(train_data_pca, train_target)
        y_pred = model.predict(test_data_pca)
        
        mse = mean_squared_error(test_target, y_pred)
        r2 = r2_score(test_target, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(test_target - y_pred))
        
        xgb_result_list.append({
            "mse": mse,
            "r2": r2,
            "rmse": rmse,
            "mae": mae,
            "fold": fold_i,
        })
        fold_i += 1
    results_xgb_optimized_dfs[dat] = pd.DataFrame(xgb_result_list)

[I 2025-08-19 19:27:57,694] A new study created in memory with name: no-name-49b3a660-2efc-4025-81fc-8f1cba3ba6d7


Starting hyperparameter optimization...

Optimizing for dataset: Ti_TM
  Optimizing SVR...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-08-19 19:27:57,728] Trial 0 finished with value: 922.33806603343 and parameters: {'C': 0.7080166777372641, 'gamma': 0.03664288488964582, 'epsilon': 0.7303805334596838}. Best is trial 0 with value: 922.33806603343.
[I 2025-08-19 19:27:57,745] Trial 1 finished with value: 920.4224004632733 and parameters: {'C': 0.716739811622706, 'gamma': 0.026605024569958235, 'epsilon': 0.009909020096150061}. Best is trial 1 with value: 920.4224004632733.
[I 2025-08-19 19:27:57,762] Trial 2 finished with value: 160.1069106516455 and parameters: {'C': 1.1072914768800126, 'gamma': 5.503780404533425e-05, 'epsilon': 0.3479369431596531}. Best is trial 2 with value: 160.1069106516455.
[I 2025-08-19 19:27:57,791] Trial 3 finished with value: 4.616285471365195 and parameters: {'C': 52.38270938257665, 'gamma': 0.0007953407290028804, 'epsilon': 0.0041235565070253995}. Best is trial 3 with value: 4.616285471365195.
[I 2025-08-19 19:27:57,813] Trial 4 finished with value: 922.9279853237564 and parameters: {

[I 2025-08-19 19:28:00,871] A new study created in memory with name: no-name-9527002d-650e-4445-b777-495ce9e39982


...

[I 2025-08-19 19:46:49,294] Trial 0 finished with value: 3.0285128127919556 and parameters: {'n_layers': 2, 'hidden_size_0': 155, 'hidden_size_1': 192, 'alpha': 7.167398116227066e-05, 'learning_rate_init': 0.016311046738317646}. Best is trial 0 with value: 3.0285128127919556.
[I 2025-08-19 19:46:50,091] Trial 1 finished with value: 6.018862591096555 and parameters: {'n_layers': 1, 'hidden_size_0': 107, 'alpha': 1.4469073799463473e-05, 'learning_rate_init': 0.03479369431596534}. Best is trial 0 with value: 3.0285128127919556.
[I 2025-08-19 19:46:51,872] Trial 2 finished with value: 5.297614436256131 and parameters: {'n_layers': 3, 'hidden_size_0': 102, 'hidden_size_1': 49, 'hidden_size_2': 12, 'alpha': 2.239817654479077e-06, 'learning_rate_init': 0.0005522897166504359}. Best is trial 0 with value: 3.0285128127919556.
[I 2025-08-19 19:46:52,535] Trial 3 finished with value: 14.991196454702086 and parameters: {'n_layers': 2, 'hidden_size_0': 127, 'hidden_size_1': 149, 'alpha': 0.000132860

[I 2025-08-19 19:50:21,249] A new study created in memory with name: no-name-05b89089-27ff-4e40-a0bc-d2eddff09bbc


[I 2025-08-19 19:50:21,245] Trial 99 finished with value: 1.9821807076124525 and parameters: {'n_layers': 2, 'hidden_size_0': 166, 'hidden_size_1': 96, 'alpha': 3.7993438209779864e-05, 'learning_rate_init': 0.000143225728911114}. Best is trial 99 with value: 1.9821807076124525.
    Best NN MSE: 1.982181
  Optimizing XGBoost...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-08-19 19:50:22,063] Trial 0 finished with value: 2.9828711986541747 and parameters: {'n_estimators': 258, 'max_depth': 9, 'learning_rate': 0.2868097549949875, 'subsample': 0.7855361528214284, 'colsample_bytree': 0.8949975776223627, 'reg_alpha': 0.3320102363966625, 'reg_lambda': 0.5110654892617195}. Best is trial 0 with value: 2.9828711986541747.
[I 2025-08-19 19:50:22,908] Trial 1 finished with value: 230.43722839355468 and parameters: {'n_estimators': 180, 'max_depth': 9, 'learning_rate': 0.2796411269270747, 'subsample': 0.7933702148684326, 'colsample_bytree': 0.6820362600374056, 'reg_alpha': 0.015032244738287792, 'reg_lambda': 0.08755316586959205}. Best is trial 0 with value: 2.9828711986541747.
[I 2025-08-19 19:50:23,954] Trial 2 finished with value: 41.51788845062256 and parameters: {'n_estimators': 161, 'max_depth': 6, 'learning_rate': 0.1878953431482189, 'subsample': 0.8915671114884505, 'colsample_bytree': 0.8123394253137142, 'reg_alpha': 0.5345027511515238, 'reg_lambda':

In [28]:
# Print comparison between default and optimized models
print("\n" + "="*80)
print("COMPARISON: DEFAULT vs OPTIMIZED MODELS")
print("="*80)

for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    print(f"\n{dat} Dataset:")
    print(f"Linear Regression    - Mean MSE: {results_entire_wave_dfs[dat]['mse'].mean():.6f}")
    print(f"SVR (default)        - Mean MSE: {results_svr_dfs[dat]['mse'].mean():.6f}")
    print(f"SVR (optimized)      - Mean MSE: {results_svr_optimized_dfs[dat]['mse'].mean():.6f}")
    print(f"Neural Net (default) - Mean MSE: {results_nn_dfs[dat]['mse'].mean():.6f}")
    print(f"Neural Net (optimized) - Mean MSE: {results_nn_optimized_dfs[dat]['mse'].mean():.6f}")
    print(f"XGBoost (default)    - Mean MSE: {results_xgb_dfs[dat]['mse'].mean():.6f}")
    print(f"XGBoost (optimized)  - Mean MSE: {results_xgb_optimized_dfs[dat]['mse'].mean():.6f}")

# Print best parameters found
print("\n" + "="*80)
print("BEST PARAMETERS FOUND")
print("="*80)

for dat in ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]:
    print(f"\n{dat}:")
    print(f"  SVR: {best_params[dat]['svr']}")
    print(f"  Neural Network: {best_params[dat]['nn']}")
    print(f"  XGBoost: {best_params[dat]['xgb']}")

print("\nOptimization complete!")
print("New result dataframes available:")
print("- results_svr_optimized_dfs")
print("- results_nn_optimized_dfs") 
print("- results_xgb_optimized_dfs")

whecho_simple("Optimization complete!")


COMPARISON: DEFAULT vs OPTIMIZED MODELS

Ti_TM Dataset:
Linear Regression    - Mean MSE: 0.000066
SVR (default)        - Mean MSE: 231.650468
SVR (optimized)      - Mean MSE: 0.003867
Neural Net (default) - Mean MSE: 0.801461
Neural Net (optimized) - Mean MSE: 0.288452
XGBoost (default)    - Mean MSE: 3.065881
XGBoost (optimized)  - Mean MSE: 1.457998

Ti_TE Dataset:
Linear Regression    - Mean MSE: 0.000105
SVR (default)        - Mean MSE: 207.143654
SVR (optimized)      - Mean MSE: 0.007182
Neural Net (default) - Mean MSE: 0.434624
Neural Net (optimized) - Mean MSE: 0.091043
XGBoost (default)    - Mean MSE: 2.557486
XGBoost (optimized)  - Mean MSE: 1.552502

Si_TE Dataset:
Linear Regression    - Mean MSE: 0.024732
SVR (default)        - Mean MSE: 294.984277
SVR (optimized)      - Mean MSE: 0.034510
Neural Net (default) - Mean MSE: 13.062227
Neural Net (optimized) - Mean MSE: 2.805551
XGBoost (default)    - Mean MSE: 3.950529
XGBoost (optimized)  - Mean MSE: 2.340006

Si_TM Dataset:


In [ ]:
# Create Excel file with model comparison and hyperparameters
def create_results_excel(results_entire_wave_dfs, results_svr_dfs, results_nn_dfs, results_xgb_dfs,
                        results_svr_optimized_dfs, results_nn_optimized_dfs, results_xgb_optimized_dfs,
                        best_params, filename='model_results.xlsx'):
    """
    Create an Excel file with model comparison and hyperparameters
    """
    
    # Sheet 1: Model Comparison
    datasets = ["Ti_TM", "Ti_TE", "Si_TE", "Si_TM"]
    
    comparison_data = {
        'Dataset': datasets,
        'Linear Regression': [results_entire_wave_dfs[dat]['mse'].mean() for dat in datasets],
        'SVR (default)': [results_svr_dfs[dat]['mse'].mean() for dat in datasets],
        'SVR (optimized)': [results_svr_optimized_dfs[dat]['mse'].mean() for dat in datasets],
        'Neural Net (default)': [results_nn_dfs[dat]['mse'].mean() for dat in datasets],
        'Neural Net (optimized)': [results_nn_optimized_dfs[dat]['mse'].mean() for dat in datasets],
        'XGBoost (default)': [results_xgb_dfs[dat]['mse'].mean() for dat in datasets],
        'XGBoost (optimized)': [results_xgb_optimized_dfs[dat]['mse'].mean() for dat in datasets]
    }
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Sheet 2: Optimal Hyperparameters
    hyperparams_data = []
    
    for dataset in datasets:
        # SVR parameters
        svr_params = best_params[dataset]['svr']
        hyperparams_data.append({
            'Dataset': dataset,
            'Model': 'SVR',
            'Parameter': 'C',
            'Value': svr_params['C']
        })
        hyperparams_data.append({
            'Dataset': dataset,
            'Model': 'SVR',
            'Parameter': 'gamma',
            'Value': svr_params['gamma']
        })
        hyperparams_data.append({
            'Dataset': dataset,
            'Model': 'SVR',
            'Parameter': 'epsilon',
            'Value': svr_params['epsilon']
        })
        
        # Neural Network parameters
        nn_params = best_params[dataset]['nn']
        hyperparams_data.append({
            'Dataset': dataset,
            'Model': 'Neural Network',
            'Parameter': 'n_layers',
            'Value': nn_params['n_layers']
        })
        
        # Hidden layer sizes
        for i in range(nn_params['n_layers']):
            hyperparams_data.append({
                'Dataset': dataset,
                'Model': 'Neural Network',
                'Parameter': f'hidden_size_{i}',
                'Value': nn_params[f'hidden_size_{i}']
            })
        
        hyperparams_data.append({
            'Dataset': dataset,
            'Model': 'Neural Network',
            'Parameter': 'alpha',
            'Value': nn_params['alpha']
        })
        hyperparams_data.append({
            'Dataset': dataset,
            'Model': 'Neural Network',
            'Parameter': 'learning_rate_init',
            'Value': nn_params['learning_rate_init']
        })
        
        # XGBoost parameters
        xgb_params = best_params[dataset]['xgb']
        xgb_param_names = ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 
                          'colsample_bytree', 'reg_alpha', 'reg_lambda']
        
        for param_name in xgb_param_names:
            hyperparams_data.append({
                'Dataset': dataset,
                'Model': 'XGBoost',
                'Parameter': param_name,
                'Value': xgb_params[param_name]
            })
    
    hyperparams_df = pd.DataFrame(hyperparams_data)
    
    # Create Excel writer and save both sheets
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # Sheet 1: Model Comparison
        comparison_df.to_excel(writer, sheet_name='Model Comparison', index=False)
        
        # Format the comparison sheet
        workbook = writer.book
        comparison_sheet = writer.sheets['Model Comparison']
        
        # Auto-adjust column widths
        for column in comparison_sheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 20)
            comparison_sheet.column_dimensions[column_letter].width = adjusted_width
        
        # Sheet 2: Hyperparameters
        hyperparams_df.to_excel(writer, sheet_name='Optimal Hyperparameters', index=False)
        
        # Format the hyperparameters sheet
        hyperparams_sheet = writer.sheets['Optimal Hyperparameters']
        
        # Auto-adjust column widths
        for column in hyperparams_sheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 25)
            hyperparams_sheet.column_dimensions[column_letter].width = adjusted_width
    
    print(f"Excel file '{filename}' created successfully!")
    print("Sheet 1: Model Comparison - Mean MSE values for all models")
    print("Sheet 2: Optimal Hyperparameters - Best parameters found by Optuna")
    
    return comparison_df, hyperparams_df

# Call the function to create the Excel file
comparison_df, hyperparams_df = create_results_excel(
    results_entire_wave_dfs, results_svr_dfs, results_nn_dfs, results_xgb_dfs,
    results_svr_optimized_dfs, results_nn_optimized_dfs, results_xgb_optimized_dfs,
    best_params, filename='suppl_table4.xlsx'
)

# # Display the dataframes for preview
# print("\nPreview of Model Comparison sheet:")
# print(comparison_df.to_string(index=False))

# print("\nPreview of Optimal Hyperparameters sheet (first 10 rows):")
# print(hyperparams_df.head(10).to_string(index=False))

Excel file 'suppl_table4.xlsx' created successfully!
Sheet 1: Model Comparison - Mean MSE values for all models
Sheet 2: Optimal Hyperparameters - Best parameters found by Optuna

Preview of Model Comparison sheet:
Dataset  Linear Regression  SVR (default)  SVR (optimized)  Neural Net (default)  Neural Net (optimized)  XGBoost (default)  XGBoost (optimized)
  Ti_TM           0.000066     231.650468         0.003867              0.801461                0.288452           3.065881             1.457998
  Ti_TE           0.000105     207.143654         0.007182              0.434624                0.091043           2.557486             1.552502
  Si_TE           0.024732     294.984277         0.034510             13.062227                2.805551           3.950529             2.340006
  Si_TM           0.103450     278.013369         0.045359              8.041610                1.982181           3.379713             1.527163

Preview of Optimal Hyperparameters sheet (first 10 rows):
D